# Analise arquivo raw produtos.csv

In [1]:
import os
import pandas as pd
from pandasql import sqldf

pysqldf = lambda q: sqldf(q, globals())

In [2]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath('.')))))
DATA_DIR = os.path.join(BASE_DIR, 'data')
RAW_DIR =  os.path.join(DATA_DIR, 'raw')

In [3]:
df = pd.read_csv(os.path.join(RAW_DIR, 'produtos.csv'), sep=';')
df.shape

(81, 6)

## Analise exploratória

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_produto   81 non-null     str    
 1   descricao    80 non-null     str    
 2   categoria    80 non-null     str    
 3   preco_custo  80 non-null     float64
 4   preco_venda  80 non-null     str    
 5   unidade      80 non-null     str    
dtypes: float64(1), str(5)
memory usage: 3.9 KB


In [5]:
df.isna().sum()

id_produto     0
descricao      1
categoria      1
preco_custo    1
preco_venda    1
unidade        1
dtype: int64

In [6]:
df.head()

,id_produto,descricao,categoria,preco_custo,preco_venda,unidade
0,P0001,Água Econômico,Bebidas,140.86,212.54,L
1,P0002,Saco plástico Premium,Embalagens,155.90,234.36,CX
2,P0003,Sabonete Premium,Higiene,133.11,81.73,L
3,P0004,Suco Premium,Bebidas,142.15,28.62,UN
4,P0005,Shampoo Premium,Higiene,147.35,112.09,KG


In [7]:
df.tail()

,id_produto,descricao,categoria,preco_custo,preco_venda,unidade
76,P0077,Filme plástico 5kg,Embalagens,16.32,12.45,L
77,P0078,Desinfetante Tradicional,Limpeza,18.26,230.97,UN
78,P0079,Detergente 12un,Limpeza,136.62,114.51,L
79,P0080,Sabão 500ml,Limpeza,79.59,195.98,UN
80,P0063,Bebida energética Tradicional,Bebidas,157.34,139.05,L


In [8]:
df.sample(5)

,id_produto,descricao,categoria,preco_custo,preco_venda,unidade
77,P0078,Desinfetante Tradicional,Limpeza,18.26,230.97,UN
12,P0013,Café Econômico,alimentos,92.27,161.05,KG
23,P0024,Saco plástico 12un,Embalagens,23.60,34.6,L
78,P0079,Detergente 12un,Limpeza,136.62,114.51,L
65,P0066,Refrigerante 12un,Bebidas,102.19,75.01,CX


In [9]:
df.isnull().sum()

id_produto     0
descricao      1
categoria      1
preco_custo    1
preco_venda    1
unidade        1
dtype: int64

In [10]:
q = '''SELECT id_produto
            , descricao
            , categoria
            , preco_custo
            , preco_venda
            , unidade
        FROM df
        WHERE descricao isnull
                    or categoria isnull
                    or preco_custo isnull
                    or preco_venda isnull
                    or unidade isnull
'''

In [11]:
print(pysqldf(q))

  id_produto                   descricao   categoria  preco_custo preco_venda  \
0      P0028  Água sanitária Tradicional     Limpeza       129.01       58.57   
1      P0032          Filme plástico 5kg  Embalagens         3.05         NaN   
2      P0045                   Café 12un   Alimentos          NaN      165.17   
3      P0058                         NaN     Limpeza        56.58       57.64   
4      P0076                Sabonete 5kg         NaN       124.58      168.84   

  unidade  
0     NaN  
1       L  
2      UN  
3      CX  
4      KG  


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_produto   81 non-null     str    
 1   descricao    80 non-null     str    
 2   categoria    80 non-null     str    
 3   preco_custo  80 non-null     float64
 4   preco_venda  80 non-null     str    
 5   unidade      80 non-null     str    
dtypes: float64(1), str(5)
memory usage: 3.9 KB


## Tratamento de dados

In [13]:
df_original = df.copy()

In [14]:
df = df.fillna('NAO INFORMADO')

In [15]:
df[['preco_custo', 'preco_venda']] = \
df[['preco_custo', 'preco_venda']].replace('NAO INFORMADO', 0)

In [16]:
df['preco_custo'] = pd.to_numeric(df['preco_custo'], errors='coerce')

In [17]:
df['preco_venda'] = pd.to_numeric(df['preco_venda'], errors='coerce')

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_produto   81 non-null     str    
 1   descricao    81 non-null     str    
 2   categoria    81 non-null     str    
 3   preco_custo  81 non-null     float64
 4   preco_venda  80 non-null     float64
 5   unidade      81 non-null     str    
dtypes: float64(2), str(4)
memory usage: 3.9 KB


In [19]:
df.sample(10)

,id_produto,descricao,categoria,preco_custo,preco_venda,unidade
68,P0069,Detergente 500ml,Limpeza,172.90,26.52,CX
26,P0027,Suco 12un,Bebidas,140.43,117.85,CX
24,P0025,Pote 12un,Embalagens,160.67,148.20,L
74,P0075,Água Tradicional,Bebidas,31.53,75.90,KG
37,P0038,Água sanitária Econômico,Limpeza,150.61,114.46,UN
5,P0006,Refrigerante Tradicional,Bebidas,119.49,97.44,KG
33,P0034,Papel higiênico 1L,Higiene,122.89,98.82,CX
14,P0015,Suco 5kg,Bebidas,99.53,3.10,KG
48,P0049,Shampoo 1L,Higiene,94.87,157.16,L
66,P0067,Açúcar 12un,Alimentos,76.08,144.28,CX


## Salvando dados de produtos em Bronze

In [20]:
BRONZE_DIR = os.path.join(DATA_DIR, 'bronze')

In [21]:
df.to_csv(os.path.join(BRONZE_DIR, 'b_produtos.csv'), index=False)